# Exploring change_ds — Why Are Some Std Devs So Large?

Replicates the `change_ds` construction from `analysis_app.py` and converts to a flat DataFrame for inspection.

In [9]:
import os, sys

# Source paths are all relative to the repo root — set cwd there
REPO_ROOT = os.path.dirname(os.path.abspath(''))   # notebooks/ -> repo root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(REPO_ROOT)

sys.path.insert(0, os.path.join(os.getcwd(), 'src'))

import numpy as np
import pandas as pd
import xarray as xr

print("Working directory:", os.getcwd())


Working directory: /Users/brooksgingrich/maine-hospital-fin-health


In [10]:
from c_Processing.c_main_data_pipeline import create_full_underived_df, to_dataset
from d_Transformations.calc_changes import calc_period_over_period_change

# Match defaults from analysis_app.py
selected_states = ['ME']
num_years_ma = 5

underived_df = create_full_underived_df(selected_states)
underived_ds = to_dataset(underived_df)

change_ds = calc_period_over_period_change(underived_ds, 'value', num_years_ma)
print(change_ds)

<xarray.Dataset> Size: 4MB
Dimensions:         (organization: 43, state: 1, measure: 103, year: 20)
Coordinates:
  * organization    (organization) object 344B 'Bridgton Hospital' ... 'York ...
  * state           (state) object 8B 'ME'
    year_failed     (organization, state) object 344B None None ... None None
  * measure         (measure) object 824B 'Accounts Payable Plus Accrued Expe...
  * year            (year) int64 160B 2005 2006 2007 2008 ... 2022 2023 2024
Data variables:
    value           (organization, state, measure, year) float64 709kB 1.198e...
    ln_value        (organization, state, measure, year) float64 709kB 14.0 ....
    ln_pct_change   (organization, state, measure, year) float64 709kB nan .....
    pct_change      (organization, state, measure, year) float64 709kB nan .....
    ma_pct_change   (organization, state, measure, year) float64 709kB nan .....
    cum_pct_change  (organization, state, measure, year) float64 709kB 0.0 .....


/Users/brooksgingrich/maine-hospital-fin-health/notebooks/../src/b_Ingest/ingest_me_financials.py:176: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  .stack(dropna=False)
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/xarray/computation/apply_ufunc.py:820: RuntimeWarning: divide by zero encountered in log
  result_data = func(*input_data)
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/xarray/computation/apply_ufunc.py:820: RuntimeWarning: invalid value encountered in log
  result_data = func(*input_data)
/opt/homebrew/Caskroom/miniconda/base/lib/python3.13/site-packages/numpy/_core/fromnumeric.py:57: RuntimeWarning: invalid value encountered in accumulate
  return bound(*args, **kwds)


## Convert to DataFrame

In [11]:
df = change_ds.to_dataframe().reset_index()
print(df.shape)
df.head(10)

(88580, 11)


,organization,state,measure,year,year_failed,value,ln_value,ln_pct_change,pct_change,ma_pct_change,cum_pct_change
0,Bridgton Hospital,ME,Accounts Payable Plus Accrued Expenses,2005,None,1197883.0,13.996066,NaN,NaN,NaN,0.000000
1,Bridgton Hospital,ME,Accounts Payable Plus Accrued Expenses,2006,None,1503921.0,14.223586,0.227520,0.255482,0.255482,0.255482
2,Bridgton Hospital,ME,Accounts Payable Plus Accrued Expenses,2007,None,2278417.0,14.638991,0.415405,0.514984,0.379143,0.902036
3,Bridgton Hospital,ME,Accounts Payable Plus Accrued Expenses,2008,None,2201956.0,14.604857,-0.034135,-0.033559,0.224987,0.838206
4,Bridgton Hospital,ME,Accounts Payable Plus Accrued Expenses,2009,None,2286439.0,14.642506,0.037650,0.038367,0.175402,0.908733
5,Bridgton Hospital,ME,Accounts Payable Plus Accrued Expenses,2010,None,2681895.0,14.802034,0.159528,0.172957,0.174912,1.238862
6,Bridgton Hospital,ME,Accounts Payable Plus Accrued Expenses,2011,None,2788165.0,14.840894,0.038860,0.039625,0.131407,1.327577
7,Bridgton Hospital,ME,Accounts Payable Plus Accrued Expenses,2012,None,2640089.0,14.786323,-0.054571,-0.053109,0.029905,1.203962
8,Bridgton Hospital,ME,Accounts Payable Plus Accrued Expenses,2013,None,3007846.0,14.916735,0.130412,0.139297,0.064362,1.510968
9,Bridgton Hospital,ME,Accounts Payable Plus Accrued Expenses,2014,None,3236640.0,14.990046,0.073312,0.076066,0.071981,1.701967


## Std dev by measure — spot large values

In [12]:
pct_std = (
    df.dropna(subset=['pct_change'])
    .groupby('measure')['pct_change']
    .agg(['mean', 'std', 'count'])
    .rename(columns={'mean': 'mean_pct_change', 'std': 'std_pct_change'})
    .sort_values('std_pct_change', ascending=False)
)
pct_std.head(20)

,mean_pct_change,std_pct_change,count
measure,,,
Due from Affiliates (Current),4478.813690,86092.064370,370
Transfers from/to Affiliates,113.335534,926.759747,69
Consolidations with Support Organizations,319.924553,783.570869,6
Net Assets Released for Restrictions,39.859389,539.255751,243
Accumulated Depreciation,18.198434,480.361923,702
Average Age of Plant,16.254221,428.462534,699
Estimated Third Party Settlements (Current),13.200125,259.328716,431
Receivables (Restricted),12.243575,134.466382,140
Return on Equity,15.724901,99.405669,469


## Drill into a high-std measure — show the extreme rows

In [22]:
top_measure = 'Net Income'
print(f'Inspecting: {top_measure}')

sub = df[df['measure'] == top_measure].dropna(subset=['pct_change'])
sub_sorted = sub.reindex(sub['pct_change'].abs().sort_values(ascending=False).index)
sub_sorted[['organization', 'state', 'year', 'value', 'ln_value', 'pct_change', 'ma_pct_change']].head(30)

Inspecting: Net Income


,organization,state,year,value,ln_value,pct_change,ma_pct_change
56587,Northern Maine Medical Center,ME,2012,16135318.0,16.596521,242.096966,242.096966
5086,Cary Medical Center,ME,2011,4172134.0,15.243938,51.669814,5.864938
75122,St. Andrews Hospital,ME,2007,1051454.0,13.865685,46.461136,46.461136
77196,St. Joseph Hospital,ME,2021,1197000.0,13.995329,40.275862,-0.266875
31866,Mount Desert Island Hospital,ME,2011,315846.0,12.663010,27.818066,5.538442
5083,Cary Medical Center,ME,2008,1309897.0,14.085459,21.442040,-0.011062
31872,Mount Desert Island Hospital,ME,2017,2207609.0,14.607421,17.083742,0.916774
35986,Northern Light AR Gould Hospital,ME,2011,3550064.0,15.082476,16.744198,0.363273
56592,Northern Maine Medical Center,ME,2017,8301542.0,15.931952,12.391441,-0.124459
62766,Penobscot Valley Hospital,ME,2011,473838.0,13.068621,10.966513,1.582862


In [24]:
sub[sub['organization'] == 'Northern Maine Medical Center'].sort_values(by='year', ascending=True)

,organization,state,measure,year,year_failed,value,ln_value,ln_pct_change,pct_change,ma_pct_change,cum_pct_change,prior_value
56581,Northern Maine Medical Center,ME,Net Income,2006,None,2712531.0,14.813393,0.687740,0.989215,0.989215,0.989215,1363619.0
56582,Northern Maine Medical Center,ME,Net Income,2007,None,2009507.0,14.513400,-0.299993,-0.259176,0.213943,0.473657,2712531.0
56587,Northern Maine Medical Center,ME,Net Income,2012,None,16135318.0,16.596521,5.493460,242.096966,242.096966,357.241601,66374.0
56588,Northern Maine Medical Center,ME,Net Income,2013,None,3472249.0,15.060313,-1.536208,-0.784804,6.232800,76.092007,16135318.0
56589,Northern Maine Medical Center,ME,Net Income,2014,None,1598294.0,14.284447,-0.775866,-0.539695,1.887705,34.485846,3472249.0
56590,Northern Maine Medical Center,ME,Net Income,2015,None,1652987.0,14.318095,0.033647,0.034220,1.233921,35.700157,1598294.0
56591,Northern Maine Medical Center,ME,Net Income,2016,None,619914.0,13.337336,-0.980758,-0.624973,0.563388,12.763533,1652987.0
56592,Northern Maine Medical Center,ME,Net Income,2017,None,8301542.0,15.931952,2.594616,12.391441,-0.124459,183.313547,619914.0
56593,Northern Maine Medical Center,ME,Net Income,2018,None,588996.0,13.286175,-2.645777,-0.929050,-0.298706,12.077082,8301542.0
56594,Northern Maine Medical Center,ME,Net Income,2019,None,570275.0,13.253874,-0.032301,-0.031785,-0.186260,11.661432,588996.0


## Check for near-zero denominators (small base values driving large % changes)

In [14]:
# Rows where pct_change is extreme but the prior-year value was tiny
# (prior value = current value / (1 + pct_change))
df['prior_value'] = df['value'] / (1 + df['pct_change'])

extreme = df[df['pct_change'].abs() > 1].copy()  # > 100% change
extreme_sorted = extreme.sort_values('pct_change', key=abs, ascending=False)
extreme_sorted[['organization', 'state', 'measure', 'year', 'prior_value', 'value', 'pct_change']].head(40)

,organization,state,measure,year,prior_value,value,pct_change
2291,Calais Regional Hospital,ME,Cash Flow to Total Debt,2016,0.000,1.500000e-02,inf
50116,Northern Light Mayo Hospital,ME,Fixed Asset Financing,2021,0.000,1.510000e+00,inf
46436,Northern Light Inland Hospital,ME,Non-Operating Margin,2021,0.000,6.090000e+00,inf
32666,Mount Desert Island Hospital,ME,Total Margin,2011,0.000,6.000000e-03,inf
28223,Miles Memorial Hospital,ME,Return on Assets,2008,0.000,4.700000e-02,inf
34069,New England Rehabilitation Hospital,ME,Non-Operating Margin,2014,0.000,1.000000e-03,inf
54678,Northern Light Sebasticook Valley Hospital,ME,Non-Operating Margin,2023,0.000,1.000000e-02,inf
28543,Miles Memorial Hospital,ME,Total Margin,2008,0.000,2.900000e-02,inf
54675,Northern Light Sebasticook Valley Hospital,ME,Non-Operating Margin,2020,0.000,1.000000e-02,inf
2736,Calais Regional Hospital,ME,Fixed Asset Financing,2021,0.000,1.328400e+02,inf


## Summary: how many extreme observations per measure?

In [15]:
extreme_counts = (
    extreme.groupby('measure')
    .size()
    .rename('n_extreme_obs')
    .sort_values(ascending=False)
)
extreme_counts.head(20)

measure
Cash Flow to Total Debt                         127
Non-Operating Margin                            126
Cash and Investments (Unrestricted)             122
Return on Equity                                109
Operating Margin                                108
Total Margin                                    101
Total Non-Operating Revenue                      98
Current Days Cash on Hand                        98
Total Investment Income                          94
Third Party Settlements Receivable               90
Total Change in Unrestricted Net Assets          90
Debt Service Coverage Ratio                      85
Due from Affiliates (Current)                    81
Other Non-Current Assets                         79
Net Operating Income                             73
Net Income                                       71
Total Surplus/Deficit                            69
Net Assets Released for Restrictions             68
Other Current Liabilities                        68
Real